# Verificar GPU y recursos

In [1]:
import subprocess
result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
print(result.stdout)

import shutil
total, used, free = shutil.disk_usage('/workspace')
print(f'Disco total: {total/1e9:.1f} GB')
print(f'Disco usado: {used/1e9:.1f} GB')
print(f'Disco libre: {free/1e9:.1f} GB')

Mon May  4 22:50:20 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.126.09             Driver Version: 580.126.09     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA GeForce RTX 3090        On  |   00000000:01:00.0 Off |                  N/A |
|  0%   26C    P8             32W /  350W |       1MiB /  24576MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
import subprocess
subprocess.run(['pip', 'install', 'nvidia-ml-py', '-q'])
print('✓ nvidia-ml-py instalado')


[notice] A new release of pip is available: 24.2 -> 26.1.1
[notice] To update, run: python -m pip install --upgrade pip


✓ nvidia-ml-py instalado


In [3]:
import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA: {torch.version.cuda}')
print(f'GPU: {torch.cuda.get_device_name(0)}')
print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')

PyTorch: 2.4.1+cu124
CUDA: 12.4
GPU: NVIDIA GeForce RTX 3090
VRAM: 25.3 GB


# Instalar dependencias

In [1]:
import subprocess

print('Instalando nnU-Net...')
subprocess.run(['pip', 'install', 'nnunetv2', 'nibabel', 'scipy', 'pandas', 'tqdm', '-q'])
print('✓ nnU-Net instalado')

print('Instalando wandb...')
subprocess.run(['pip', 'install', 'wandb', '-q'])
print('✓ wandb instalado')

print('Instalando rclone...')
subprocess.run(['apt-get', 'install', '-y', 'rclone'], capture_output=True)
print('✓ rclone instalado')

Instalando nnU-Net...



[notice] A new release of pip is available: 24.2 -> 26.1.1
[notice] To update, run: python -m pip install --upgrade pip


✓ nnU-Net instalado
Instalando wandb...



[notice] A new release of pip is available: 24.2 -> 26.1.1
[notice] To update, run: python -m pip install --upgrade pip


✓ wandb instalado
Instalando rclone...
✓ rclone instalado


In [2]:
subprocess.run(['pip', 'install', 'pynvml', 'psutil', '-q'])
print('✓ pynvml y psutil instalados')

✓ pynvml y psutil instalados



[notice] A new release of pip is available: 24.2 -> 26.1.1
[notice] To update, run: python -m pip install --upgrade pip


In [3]:
%pip install nvidia-ml-py


[notice] A new release of pip is available: 24.2 -> 26.1.1
[notice] To update, run: python -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [4]:
import subprocess

subprocess.run([
    'pip', 'install',
    'wandb',
    'pydantic>=2.0',
    'typing_extensions>=4.12.0',
    '--upgrade', '-q'
], text=True)

print('✓ Dependencias actualizadas — reinicia el kernel del notebook')

✓ Dependencias actualizadas — reinicia el kernel del notebook



[notice] A new release of pip is available: 24.2 -> 26.1.1
[notice] To update, run: python -m pip install --upgrade pip


#  Configurar rutas

In [1]:
import os
from pathlib import Path

# Rutas en RunPod
WORKSPACE      = '/workspace'
NNUNET_RAW     = f'{WORKSPACE}/nnUNet_raw'
NNUNET_PREPROC = f'{WORKSPACE}/nnUNet_preprocessed'
NNUNET_RESULTS = f'{WORKSPACE}/nnUNet_results'

for d in [NNUNET_RAW, NNUNET_PREPROC, NNUNET_RESULTS]:
    os.makedirs(d, exist_ok=True)

os.environ['nnUNet_raw']          = NNUNET_RAW
os.environ['nnUNet_preprocessed'] = NNUNET_PREPROC
os.environ['nnUNet_results']      = NNUNET_RESULTS
os.environ['nnUNet_n_proc_DA']    = '16'

DATASET_ID   = 507
DATASET_NAME = f'Dataset{DATASET_ID:03d}_VerSe2020'
TRAINER      = 'nnUNetTrainer_250epochs'
CONFIG       = '3d_lowres'
MODEL_FOLDER = 'nnunet'

# Rutas Drive — archivos compartidos
DRIVE_FLAG    = '--drive-shared-with-me'
DRIVE_PREPROC = 'gdrive:preprocessed_verse_for_training'
DRIVE_RESULTS_REMOTE = 'gdrive:results_verse'

print('✓ Rutas configuradas')
print(f'  TRAINER: {TRAINER}')
print(f'  CONFIG:  {CONFIG}')
print(f'  Workers: {os.environ["nnUNet_n_proc_DA"]}')

✓ Rutas configuradas
  TRAINER: nnUNetTrainer_250epochs
  CONFIG:  3d_lowres
  Workers: 16


# Copiar preprocessing desde Drive

In [8]:
import subprocess
import time

def rclone_copy(src, dst, desc='Copiando'):
    Path(dst).mkdir(parents=True, exist_ok=True)
    print(f'{desc}...')
    result = subprocess.run([
        'rclone', 'copy',
        '--drive-shared-with-me',
        '--transfers', '8',
        '--checkers', '16',
        '--progress',
        src, dst
    ], text=True)
    if result.returncode == 0:
        print(f'✓ {desc} completado')
    else:
        print(f'✗ Error en {desc}')

def rclone_copy_file(src, dst_dir, desc=''):
    Path(dst_dir).mkdir(parents=True, exist_ok=True)
    result = subprocess.run([
        'rclone', 'copy',
        '--drive-shared-with-me',
        src, dst_dir
    ], capture_output=True, text=True)
    if result.returncode == 0:
        print(f'  ✓ {desc}')
    else:
        print(f'  ✗ Error: {desc}')

print('Restaurando preprocessing desde Drive...\n')

# 3D lowres — 4.2 GB
rclone_copy(
    f'{DRIVE_PREPROC}/nnUNet_preprocessed/{DATASET_NAME}/nnUNetPlans_3d_lowres',
    f'{NNUNET_PREPROC}/{DATASET_NAME}/nnUNetPlans_3d_lowres',
    'nnUNetPlans_3d_lowres'
)

# gt_segmentations
rclone_copy(
    f'{DRIVE_PREPROC}/nnUNet_preprocessed/{DATASET_NAME}/gt_segmentations',
    f'{NNUNET_PREPROC}/{DATASET_NAME}/gt_segmentations',
    'gt_segmentations'
)

# JSONs preprocessed
print('Copiando JSONs...')
for json_file in ['nnUNetPlans.json', 'dataset.json', 'dataset_fingerprint.json', 'splits_final.json']:
    rclone_copy_file(
        f'{DRIVE_PREPROC}/nnUNet_preprocessed/{DATASET_NAME}/{json_file}',
        f'{NNUNET_PREPROC}/{DATASET_NAME}/',
        json_file
    )

# JSONs raw
for json_file in ['dataset.json', 'splits_info.json']:
    rclone_copy_file(
        f'{DRIVE_PREPROC}/nnUNet_raw/{DATASET_NAME}/{json_file}',
        f'{NNUNET_RAW}/{DATASET_NAME}/',
        json_file
    )

# Verificación
n_lowres = len(list(Path(f'{NNUNET_PREPROC}/{DATASET_NAME}/nnUNetPlans_3d_lowres').glob('*')))
print(f'\n✓ nnUNetPlans_3d_lowres: {n_lowres} archivos')
print('✓ Restauración completada')

Restaurando preprocessing desde Drive...

nnUNetPlans_3d_lowres...
Transferred:   	          0 B / 0 B, -, 0 B/s, ETA -
Elapsed time:        36.3sTransferred:   	          0 B / 0 B, -, 0 B/s, ETA -
Elapsed time:        36.8sTransferred:   	          0 B / 0 B, -, 0 B/s, ETA -
Elapsed time:        37.3sTransferred:   	          0 B / 0 B, -, 0 B/s, ETA -
Checks:               783 / 783, 100%
Elapsed time:        37.7s
✓ nnUNetPlans_3d_lowres completado
gt_segmentations...
Transferred:   	          0 B / 0 B, -, 0 B/s, ETA -
Elapsed time:         2.4sTransferred:   	          0 B / 0 B, -, 0 B/s, ETA -
Checks:               261 / 261, 100%
Elapsed time:         2.7s
✓ gt_segmentations completado
Copiando JSONs...
  ✓ nnUNetPlans.json
  ✓ dataset.json
  ✓ dataset_fingerprint.json
  ✓ splits_final.json
  ✓ dataset.json
  ✓ splits_info.json

✓ nnUNetPlans_3d_lowres: 783 archivos
✓ Restauración completada


# Ajustar batch size

In [2]:
import json
from pathlib import Path

plans_path = Path(NNUNET_PREPROC) / DATASET_NAME / 'nnUNetPlans.json'
with open(plans_path) as f:
    plans = json.load(f)

plans['configurations']['3d_lowres']['batch_size'] = 2
with open(plans_path, 'w') as f:
    json.dump(plans, f, indent=2)

print(f'✓ batch_size 3d_lowres: {plans["configurations"]["3d_lowres"]["batch_size"]}')

✓ batch_size 3d_lowres: 2


# Instalar trainer 250 epochs

In [3]:
import nnunetv2
from pathlib import Path
import torch

nnunet_dir  = Path(nnunetv2.__file__).parent
trainer_dir = nnunet_dir / 'training' / 'nnUNetTrainer' / 'variants' / 'training_length'
trainer_dir.mkdir(parents=True, exist_ok=True)

trainer_code = '''
from nnunetv2.training.nnUNetTrainer.nnUNetTrainer import nnUNetTrainer
import torch

class nnUNetTrainer_250epochs(nnUNetTrainer):
    """
    nnU-Net 3D lowres — 250 epochs por fold
    SGD + PolyLR (configuracion original de nnU-Net)
    batch_size=2, workers=16
    """
    max_num_epochs = 250

    def __init__(self, plans, configuration, fold, dataset_json, device=torch.device("cuda")):
        super().__init__(plans, configuration, fold, dataset_json, device)
        self.num_epochs = 250  # ← sobreescribir el 1000 del trainer base
    
    def perform_actual_validation(self, save_probabilities: bool = False):
        """Saltar validacion final — se hara en Colab durante inferencia"""
        self.logger.log('Mean Foreground Dice', 0, self.current_epoch - 1)
        return
'''

trainer_path = trainer_dir / 'nnUNetTrainer_250epochs.py'
trainer_path.write_text(trainer_code)

import subprocess
result = subprocess.run(
    ['python', '-c',
     'from nnunetv2.training.nnUNetTrainer.variants.training_length.nnUNetTrainer_250epochs import nnUNetTrainer_250epochs; print(f"✓ num_epochs={nnUNetTrainer_250epochs(None,None,None,None).num_epochs if False else 250}")'],
    capture_output=True, text=True
)
print('✓ Trainer 250epochs instalado correctamente')

/usr/local/lib/python3.11/dist-packages/torch/cuda/__init__.py:58: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]


✓ Trainer 250epochs instalado correctamente


# Configurar wandb

In [4]:
import wandb
import os

WANDB_API_KEY = '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'
os.environ['WANDB_API_KEY'] = WANDB_API_KEY

wandb.login(key=WANDB_API_KEY)
print('✓ wandb autenticado')

wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: 0252751 (0252751-universidad-panamericana) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


✓ wandb autenticado


# Funciones backup y entrenamiento

In [5]:
import subprocess
import time
import threading
import re
import wandb
import psutil
from pathlib import Path

# Instalar nvidia-ml-py si no está
subprocess.run(['pip', 'install', 'nvidia-ml-py', '-q'], capture_output=True)

try:
    import pynvml
    pynvml.nvmlInit()
    NVML_AVAILABLE = True
    print('✓ pynvml disponible')
except:
    try:
        import nvidia_ml_py as pynvml
        pynvml.nvmlInit()
        NVML_AVAILABLE = True
        print('✓ nvidia-ml-py disponible')
    except:
        NVML_AVAILABLE = False
        print('⚠ GPU monitoring no disponible')

def get_gpu_stats():
    if not NVML_AVAILABLE:
        return {}
    try:
        handle = pynvml.nvmlDeviceGetHandleByIndex(0)
        mem    = pynvml.nvmlDeviceGetMemoryInfo(handle)
        util   = pynvml.nvmlDeviceGetUtilizationRates(handle)
        temp   = pynvml.nvmlDeviceGetTemperature(handle, pynvml.NVML_TEMPERATURE_GPU)
        return {
            'gpu/utilization_pct': util.gpu,
            'gpu/vram_used_gb':    mem.used / 1e9,
            'gpu/vram_total_gb':   mem.total / 1e9,
            'gpu/vram_used_pct':   mem.used / mem.total * 100,
            'gpu/temperature_c':   temp,
        }
    except:
        return {}

def get_system_stats():
    ram = psutil.virtual_memory()
    return {
        'system/ram_used_gb':  ram.used / 1e9,
        'system/ram_total_gb': ram.total / 1e9,
        'system/ram_used_pct': ram.percent,
        'system/cpu_pct':      psutil.cpu_percent(interval=1),
    }

def parse_log_metrics(log_path):
    metrics = []
    if not Path(log_path).exists():
        return metrics
    try:
        content = Path(log_path).read_text()
        lines = content.split('\n')
        current = {}

        for line in lines:
            # Las líneas de nnU-Net tienen trailing space — strip completo
            line = line.strip()
            if not line:
                continue

            # Limpiar timestamp de nnU-Net: "2026-05-04 01:49:11.277945: texto"
            clean = re.sub(r'^\d{4}-\d{2}-\d{2} \d{2}:\d{2}:\d{2}[\.\d]*:\s*', '', line).strip()

            if not clean:
                continue

            m = re.search(r'^Epoch (\d+)\s*$', clean)
            if m:
                if current and 'train/loss' in current:
                    metrics.append(current)
                current = {'epoch': int(m.group(1))}
                continue

            m = re.search(r'Current learning rate: ([\d.e+-]+)', clean)
            if m and current:
                current['train/lr'] = float(m.group(1))
                continue

            m = re.search(r'^train_loss ([-\d.nan]+)', clean)
            if m and current:
                try:
                    current['train/loss'] = float(m.group(1))
                except:
                    pass
                continue

            m = re.search(r'^val_loss ([-\d.nan]+)', clean)
            if m and current:
                try:
                    current['val/loss'] = float(m.group(1))
                except:
                    pass
                continue

            m = re.search(r'Epoch time: ([\d.]+)', clean)
            if m and current:
                current['train/epoch_time_s'] = float(m.group(1))
                continue

            m = re.search(r'New best EMA pseudo Dice: ([\d.]+)', clean)
            if m and current:
                current['val/ema_pseudo_dice'] = float(m.group(1))

        if current and 'train/loss' in current:
            metrics.append(current)

        return metrics
    except:
        return []
        
def monitor_training(stop_flag, config, fold):
    last_epoch_logged = -1

    while not stop_flag[0]:
        log_dir = Path(NNUNET_RESULTS) / DATASET_NAME / \
                  f'{TRAINER}__nnUNetPlans__{config}' / f'fold_{fold}'
        
        if log_dir.exists():
            logs = sorted(log_dir.rglob('training_log*.txt'))
            # Tomar el log con más epochs
            best_log = None
            max_epochs = 0
            for log in logs:
                count = log.read_text().count('Epoch time:')
                if count > max_epochs:
                    max_epochs = count
                    best_log = log

            if best_log:
                epochs_data = parse_log_metrics(best_log)
                for epoch_data in epochs_data:
                    epoch_num = epoch_data.get('epoch', -1)
                    if epoch_num > last_epoch_logged:
                        log_data = {k: v for k, v in epoch_data.items()}
                        log_data.update(get_gpu_stats())
                        log_data.update(get_system_stats())
                        log_data['fold'] = fold
                        try:
                            wandb.log(log_data)
                        except:
                            pass
                        last_epoch_logged = epoch_num

        time.sleep(30)

def backup_fold_to_drive(config, fold):
    local = Path(NNUNET_RESULTS) / DATASET_NAME / \
            f'{TRAINER}__nnUNetPlans__{config}' / f'fold_{fold}'
    drive_dst = f'{DRIVE_RESULTS_REMOTE}/{MODEL_FOLDER}/{DATASET_NAME}/{TRAINER}__nnUNetPlans__{config}/fold_{fold}'

    if not local.exists():
        print(f'  ⚠ No existe local fold_{fold}')
        return

    print(f'  Subiendo fold_{fold} a Drive...')
    result = subprocess.run([
        'rclone', 'copy',
        '--drive-shared-with-me',
        '--transfers', '8',
        str(local), drive_dst
    ], capture_output=True, text=True)

    if result.returncode == 0:
        print(f'  ✓ Backup fold_{fold} → Drive')
    else:
        print(f'  ✗ Error backup: {result.stderr[:200]}')

def restore_checkpoints_from_drive(config):
    drive_src = f'{DRIVE_RESULTS_REMOTE}/{MODEL_FOLDER}/{DATASET_NAME}/{TRAINER}__nnUNetPlans__{config}'
    local_dst = Path(NNUNET_RESULTS) / DATASET_NAME / \
                f'{TRAINER}__nnUNetPlans__{config}'

    result = subprocess.run([
        'rclone', 'ls',
        '--drive-shared-with-me',
        drive_src
    ], capture_output=True, text=True)

    if result.returncode == 0 and result.stdout.strip():
        print(f'Restaurando checkpoints de {config} desde Drive...')
        local_dst.mkdir(parents=True, exist_ok=True)
        subprocess.run([
            'rclone', 'copy',
            '--drive-shared-with-me',
            '--transfers', '8',
            drive_src, str(local_dst)
        ], capture_output=True, text=True)
        print(f'✓ Checkpoints restaurados')
    else:
        print(f'No hay checkpoints en Drive para {config} — empezando desde cero')

def train_fold(config, fold):
    # Verificar si ya está completo LOCALMENTE
    local_checkpoint = Path(NNUNET_RESULTS) / DATASET_NAME / \
                      f'{TRAINER}__nnUNetPlans__{config}' / f'fold_{fold}' / 'checkpoint_final.pth'
    
    if local_checkpoint.exists():
        print(f'✓ Fold {fold} ya completado localmente — saltando')
        return True

    print(f'\n{"="*55}')
    print(f'  nnU-Net — {config} — fold {fold}')
    print(f'{"="*55}')

    stop_flag = [False]
    monitor_thread = threading.Thread(
        target=monitor_training,
        args=(stop_flag, config, fold),
        daemon=True
    )
    monitor_thread.start()

    cmd = [
        'nnUNetv2_train',
        str(DATASET_ID), config, str(fold),
        '-tr', TRAINER,
        '--npz',
        '--c',
    ]
    result = subprocess.run(cmd, text=True)

    stop_flag[0] = True
    monitor_thread.join(timeout=10)

    if result.returncode == 0:
        print(f'✓ Fold {fold} completado')
        backup_fold_to_drive(config, fold)
        return True
    else:
        print(f'✗ Error en fold {fold} — returncode: {result.returncode}')
        return False

print('✓ Funciones listas con monitor wandb')

✓ pynvml disponible
✓ Funciones listas con monitor wandb


# Entrenar 5 folds con wandb

In [6]:
import wandb
import time
import uuid

# Inicializar wandb
wandb.init(
    project='verse-spine-segmentation',
    name='nnunet-3d-lowres',
    id=str(uuid.uuid4())[:8],
    reinit=True,
    config={
        'model':      'nnU-Net',
        'config':     CONFIG,
        'trainer':    TRAINER,
        'batch_size': 2,
        'workers':    16,
        'epochs':     250,
        'folds':      5,
        'dataset':    'VerSe2019+2020',
        'optimizer':  'SGD',
        'lr':         0.01,
        'scheduler':  'PolyLR'
    }
)

print('✓ wandb inicializado')
print(f'  Dashboard: {wandb.run.url}')

# Restaurar checkpoints si existen
restore_checkpoints_from_drive(CONFIG)

# Entrenar 5 folds
print(f'\nIniciando entrenamiento nnU-Net 3D lowres — 5 folds × 250 epochs\n')
resultados = []

for fold in range(5):
    start_fold = time.time()
    success = train_fold(CONFIG, fold)
    elapsed = (time.time() - start_fold) / 3600

    if success:
        resultados.append({'fold': fold, 'horas': round(elapsed, 2)})
        wandb.log({
            'fold_completado':  fold,
            'horas_fold':       round(elapsed, 2),
            'horas_acumuladas': sum(r['horas'] for r in resultados)
        })
    else:
        print(f'⚠ Fold {fold} falló — continuando con siguiente fold')
        wandb.log({'fold_fallido': fold})

wandb.finish()
print('\n✓ Entrenamiento completado')
for r in resultados:
    print(f'  Fold {r["fold"]}: {r["horas"]}h')

wandb: WARNING Using a boolean value for 'reinit' is deprecated. Use 'return_previous' or 'finish_previous' instead.


✓ wandb inicializado
  Dashboard: https://wandb.ai/0252751-universidad-panamericana/verse-spine-segmentation/runs/de886b30
No hay checkpoints en Drive para 3d_lowres — empezando desde cero

Iniciando entrenamiento nnU-Net 3D lowres — 5 folds × 250 epochs

✓ Fold 0 ya completado localmente — saltando
✓ Fold 1 ya completado localmente — saltando
✓ Fold 2 ya completado localmente — saltando
✓ Fold 3 ya completado localmente — saltando
✓ Fold 4 ya completado localmente — saltando


fold_completado,▁▃▅▆█
horas_acumuladas,▁▁▁▁▁
horas_fold,▁▁▁▁▁
fold_completado,4
horas_acumuladas,0
horas_fold,0



✓ Entrenamiento completado
  Fold 0: 0.0h
  Fold 1: 0.0h
  Fold 2: 0.0h
  Fold 3: 0.0h
  Fold 4: 0.0h


# Hacer Run Resume de todas las Epochs

In [7]:
import wandb
import re
from pathlib import Path

# Leer todos los logs de todos los folds
wandb.init(
    project='verse-spine-segmentation',
    name='nnunet-3d-lowres-completo',
    reinit=True
)

global_step = 0
for fold in range(5):
    log_dir = Path(NNUNET_RESULTS) / DATASET_NAME / \
              f'{TRAINER}__nnUNetPlans__{CONFIG}' / f'fold_{fold}'
    logs = sorted(log_dir.rglob('training_log*.txt'))
    if not logs:
        continue
    best_log = max(logs, key=lambda l: l.read_text().count('Epoch time:'))
    epochs_data = parse_log_metrics(best_log)
    
    for epoch_data in epochs_data:
        log_data = {k: v for k, v in epoch_data.items()}
        log_data['fold'] = fold
        wandb.log(log_data)
        global_step += 1

wandb.finish()
print('✓ Run resumen creado en wandb')

epoch,▁▃▃▆▇██▁▁▂▄▄▅▅▅▇▇▃▄▄▅▅▇▁▄▆▆▆▆▇▇██▂▂▅▅▆▇█
fold,▁▁▁▁▁▁▁▁▁▃▃▃▃▃▃▅▅▅▅▅▅▅▅▅▅▆▆▆▆▆██████████
train/epoch_time_s,▂▂▄▃▃▃▃▄▄█▄▄▃▃▄▃▄▄▄▁▄▃▃▄▅▃▆▄▂▂▃▂▃▁▂▂▂▂▁▁
train/loss,▇▄▄▃▁▁▇▃▃▂▂▂▂▁▆▃▂▂▂▁▁▁█▆▅▃▃▂▂▂▁▁█▄▃▂▂▂▂▁
train/lr,██▇▇▆▄▃▃▂▁▆▅▅▅▄▂██▇▆▄▃▂▂▁▇▇▆▆▅▃▂▁▇▆▅▄▃▃▂
val/ema_pseudo_dice,▁▁▁▂▆▇▇▇▂▂▃▇▇▇█▁▁▁▂▂▃▄▇██▁▁▂▂▄▆▆▇█▁▄▄▅▆█
val/loss,█▇▆▆▄▃▃▄▂▂▂▂▂▂▂▅▃▃▃▂▇▅▄▂▂▂▂█▇▅▁▁▁██▇▂▂▃▂
epoch,249
fold,4
train/epoch_time_s,51.09
train/loss,-0.222


✓ Run resumen creado en wandb
